# 05 · Consistency branch (CLIP zero-shot + SmolVLM + calibration)

**Accelerator:** GPU **T4 x2** · **Attach:** `abo-prepared`, `lv-features-checkpoints`, the `abo-fakes-*` datasets, optional `phone-set`, `hand-labels` · **Output dataset name:** `lv-consistency`

For every real listing with a trusted label we ask CLIP and SmolVLM-500M (multiple choice, letter read from
next-token logits) and cache the option probabilities. Then synthetic mismatch pairs train TWO calibrators
(with and without VLM features; the demo picks the matching one), evaluated against raw CLIP-argmax and raw
VLM-argmax, per attribute, plus the phone split and the hand labels when present.

**VLM pilot check first:** look at 20 development images and compare the letter-logit answer with a generated
answer; both should agree most of the time and the distributions must be finite and normalised.

## 0. Setup (same cell in every notebook)

**Before running:** attach the code as a Kaggle Dataset named `listing-verifier-code`
(upload the repo folder without `data/`, `outputs/`, `web/node_modules/`), **or** set `REPO_URL`
to your GitHub repo. Turn on **Internet** (pip, ABO and HF downloads). For the GPU notebooks pick
**Settings → Accelerator → GPU T4 x2**. Store your Hugging Face token as the Kaggle secret `HF_TOKEN`.

In [ ]:
import os, shutil, subprocess, sys, pathlib, json
REPO_URL = "https://github.com/kvamsi-iisc/DLVA-Final-Project-Forensic-Verification-of-Marketplace-Listings.git"   # used if the code dataset is not attached
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / "repo"; DATA = WORK / "data"; OUT = WORK / "outputs"
code_ds = pathlib.Path("/kaggle/input/listing-verifier-code")
if not REPO.exists():
    if code_ds.exists():
        shutil.copytree(code_ds, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
os.environ["LV_DATA_ROOT"] = str(DATA); os.environ["LV_OUTPUT_ROOT"] = str(OUT)
os.environ["HF_HOME"] = str(WORK / "hf_cache")           # model weights land here (removed before publishing)
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "(fine unless a gated model is needed)")
print("python", sys.version.split()[0])
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], capture_output=True, text=True)
if pip.returncode != 0:
    print(pip.stdout[-3000:], pip.stderr[-3000:])          # show WHY before stopping (e.g. a pin without a wheel for this Python)
    raise SystemExit("pip install failed; see output above")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

LOGS = OUT / "logs"; LOGS.mkdir(parents=True, exist_ok=True)

def run(*args):
    """Run a pipeline script; stream its output to the cell AND to OUT/logs/<script>_<time>.log,
    append a completion record to OUT/logs/completed.jsonl, and stop the notebook on a non-zero exit."""
    import time, datetime
    cmd = [sys.executable, *[str(a) for a in args]]
    name = pathlib.Path(str(args[0])).stem
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    log_path = LOGS / f"{name}_{stamp}.log"
    print(">>>", " ".join(cmd[1:]), "->", log_path.name, flush=True)
    t0 = time.time()
    with open(log_path, "w") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = proc.wait()
    record = {"script": name, "args": cmd[2:], "returncode": rc, "seconds": round(time.time() - t0, 1),
              "log": str(log_path), "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
    with open(LOGS / "completed.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def attach(slug, sub, dest, must_contain=None):
    """Copy a sub-folder of an attached Kaggle dataset into DATA/OUT (copy, not symlink: scripts write next to inputs).
    Refuses silently stale copies: if `must_contain` is missing from an existing dest, the copy is redone."""
    base = pathlib.Path("/kaggle/input") / slug
    src = base / sub if (base / sub).exists() else base
    dst = pathlib.Path(dest)
    if dst.exists() and (must_contain is None or (dst / must_contain).exists()):
        print("exists:", dst); return
    if dst.exists():
        shutil.rmtree(dst)
    if not src.exists():
        raise FileNotFoundError(f"dataset {slug} is not attached (expected {src})")
    print("copy", src, "->", dst); shutil.copytree(src, dst)

run("scripts/env_report.py", "--out", OUT / "results" / "env_report.json")
import torch; print(f"visible GPUs: {torch.cuda.device_count()}")


In [ ]:
attach("abo-prepared", "data/processed", DATA / "processed", must_contain="abo_clean.csv")
attach("lv-features-checkpoints", "data/features", DATA / "features", must_contain="features_q0.npz")
attach("lv-features-checkpoints", "outputs/checkpoints", OUT / "checkpoints", must_contain="main")
FAKE_SLUGS = ['abo-fakes-pilot', 'abo-fakes-turbo-t2i', 'abo-fakes-turbo-i2i', 'abo-fakes-sd15-i2i', 'abo-fakes-kand-t2i', 'abo-fakes-kand-i2i']
for slug in FAKE_SLUGS:
    src = pathlib.Path("/kaggle/input") / slug
    if not src.exists():
        print("not attached:", slug); continue
    fakes_root = next(p for p in [src / "data" / "fakes", src / "fakes", src] if p.exists())
    shutil.copytree(fakes_root, DATA / "fakes", dirs_exist_ok=True); print("merged", slug)
print("fake chunks:", len(list((DATA / "fakes").rglob("chunk_*.csv"))))
if pathlib.Path("/kaggle/input/phone-set").exists(): attach("phone-set", "phone", DATA / "phone", must_contain="labels.csv")
run("scripts/05_build_index.py", "--no-audit")     # rebuild so paths match this session (phone included)

In [ ]:
# VLM pilot: letter-logit answers vs generated answers on 20 train images
import pandas as pd
sys.path.insert(0, str(REPO))
from listing_verifier.features.vlm import SmolVLM
from listing_verifier.features.image_ops import canonicalise, load_rgb
from listing_verifier.models.consistency import VLM_QUESTIONS, vlm_options
idx = pd.read_csv(DATA / "processed" / "index.csv", low_memory=False)
dev = idx[(idx["split"] == "train") & (idx["label"] == 0) & idx["colour"].notna()].sample(20, random_state=0)
vlm = SmolVLM(device="cuda:0"); agree = 0
for _, r in dev.iterrows():
    img = canonicalise(load_rgb(r["path"])); opts = vlm_options("colour")
    p = vlm.option_probabilities(img, VLM_QUESTIONS["colour"], opts); g, txt = vlm.generate_answer(img, VLM_QUESTIONS["colour"], opts)
    agree += int(g is not None and opts[g] == opts[int(p.argmax())])
    print(f"{r['colour']:8s} logit->{opts[int(p.argmax())]:8s} ({p.max():.2f})  generated->{txt.strip()[:12]!r}")
print(f"logit/generation agreement: {agree}/20 (expect most to agree; if not, inspect the prompt format)")
del vlm

In [ ]:
run("scripts/09_run_consistency.py", "--splits", "train,val,test,phone", "--max-train", 2500, "--gpus", "auto")

In [ ]:
hl = pathlib.Path("/kaggle/input/hand-labels/hand_labels_filled.csv")
if hl.exists():
    (DATA / "hand_labels").mkdir(exist_ok=True); shutil.copy(hl, DATA / "hand_labels" / "hand_labels_filled.csv")
run("scripts/10_train_eval_consistency.py", "--tag", "main")
run("scripts/11_make_handlabel_sheet.py", "--n", 100)

### Hand labels
Download `data/hand_labels/` (the contact sheet embeds thumbnails, so it opens on any laptop). Fill `human_label` (1 = the metadata claim matches the photo, 0 = it does not) and `visible_count` where asked, upload as dataset `hand-labels` (`hand_labels_filled.csv`) and re-run this notebook. These 100 listings are a held-out audit: never use them to tune anything.

In [ ]:
from IPython.display import Image as IImage, display
r = json.load(open(OUT / "results" / "consistency" / "results.json"))
for section in ("synthetic_test", "phone_synthetic", "hand_labelled"):
    if section not in r: continue
    print(f"\n== {section} ==")
    res = r[section] if section != "synthetic_test" else r[section]["calibrated"]
    print({a: f"n={v['n']} F1mis={v['f1_mismatch']:.2f} acc={v['accuracy']:.2f} ece={v.get('ece', float('nan')):.3f}" for a, v in res.items() if isinstance(v, dict)})
if "synthetic_test" in r:
    for name, res in r["synthetic_test"].items(): print(f"{name:22s}", {a: round(v["f1_mismatch"], 2) for a, v in res.items()})
display(IImage(str(OUT / "results" / "consistency" / "f1_by_attribute.png")))

## Persisting outputs
When this notebook finishes under **Save Version → Save & Run All (Commit)**, everything in `/kaggle/working`
except the model cache is kept as the notebook output. Then: *Output tab → "New Dataset"* and name it as stated
at the top of this notebook. The next notebook attaches that dataset. The cell below checks the artifacts this
notebook promised actually exist, then removes the model cache so the output stays small.

In [ ]:
required = ["outputs/checkpoints/main/calibrator.npz", "outputs/checkpoints/main/calibrator_clip_only.npz", "outputs/results/consistency/results.json"]
run("scripts/verify_outputs.py", *[WORK / r for r in required])   # exits non-zero on a missing/empty artifact
shutil.rmtree(WORK / "hf_cache", ignore_errors=True)
shutil.rmtree(REPO / "web" / "node_modules", ignore_errors=True)
print(subprocess.run(["du", "-sh", str(DATA), str(OUT)], capture_output=True, text=True).stdout)